In [ ]:
import os
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from datasets import Dataset
import EIDA


tokenizer = GPT2Tokenizer.from_pretrained("gpt2") # 사전 크기: 50257
model = GPT2LMHeadModel.from_pretrained("gpt2_e2e_model/checkpoint-7887", device_map='auto') # layer 12개, d_model 768차원
tokenizer.pad_token = tokenizer.eos_token

# E2E NLG Challenge 데이터셋의 예시:
# name : Blue Spice | Type : coffee shop | area : city centre||A coffee shop in the city centre area called Blue Spice .
# '||' 이전 부분이 모델에 입력으로 주어지는 context(meaning representation 속성)
# '||' 이후 부분이 모델이 생성할 자연어 completion(human_reference 속성)
# E2E 폴더에 train.txt, valid.txt, test.txt 파일이 들어있어야 실행됨.
data = []
with open(os.path.join("E2E", "train.txt"), 'r', encoding='utf8') as f:
	for line in f:
		meaning_representation, human_reference = line.strip().split('||')
		data.append({'meaning_representation': meaning_representation, 'human_reference': human_reference})
train_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "valid.txt"), 'r', encoding='utf8') as f:
	for line in f:
		meaning_representation, human_reference = line.strip().split('||')
		data.append({'meaning_representation': meaning_representation, 'human_reference': human_reference})
valid_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "test.txt"), 'r', encoding='utf8') as f:
	for line in f:
		meaning_representation, human_reference = line.strip().split('||')
		data.append({'meaning_representation': meaning_representation, 'human_reference': human_reference})
test_dataset = Dataset.from_list(data)

In [2]:
max_seq_len = 144 # context와 completion을 다 담기에 충분히 긴 길이. 128로 설정하면 test set에서 잘리는 행이 생김.

tokenized_data = []
for example in train_dataset:
    tokenized_meaning_representation = tokenizer.encode(example['meaning_representation'] + tokenizer.bos_token)
    tokenized_human_reference = tokenizer.encode(' ' + example['human_reference'] + tokenizer.eos_token)
    # GPT2의 input을 (meaning representation + BOS + ' ' + human reference + EOS)로 구성하는 것은 microsoft/LoRA 깃헙에서 입력을 구성한 방식을 그대로 따랐음.
    # GPT2에서는 BOS와 EOS 둘 다 50256번 토큰 <|endoftext|>으로 되어있고, PAD 토큰은 별도로 지정되어 있지 않음.
    # 현재 PAD 토큰을 50256으로 설정해서 사용하고 있음. padding한 위치들에는 labels값 -100을 설정해서 학습과정에 관여하지 않게 만들 것이므로 PAD 토큰을 50256으로 설정한 것이 학습에 영향을 미치지 않음.

    input_ids = tokenized_meaning_representation + tokenized_human_reference + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))] # context + completion + padding 구성으로 길이 128의 시퀀스

    labels = [-100 for _ in tokenized_meaning_representation] + tokenized_human_reference + [-100 for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))]
    # labels에 -100이 들어있는 자리는 loss function이 무시하도록 설정되어 있다.(이것은 허깅페이스 transformers 패키지의 설정이 아니고 torch.nn.CrossEntropyLoss의 디폴트 설정임)
    # 모델이 context(meaning representation)를 입력받아 completion(human reference)을 생성하는 학습을 수행할 것이므로, labels에서 completion 부분의 토큰들만 남겨놓고 context 부분과 padding 부분은 -100으로 설정해주어야 의도한 학습을 수행하게 된다.
    
    tokenized_data.append({'input_ids': input_ids, 'labels': labels})
preprocessed_train_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in valid_dataset:
    tokenized_meaning_representation = tokenizer.encode(example['meaning_representation'] + tokenizer.bos_token)
    tokenized_human_reference = tokenizer.encode(' ' + example['human_reference'] + tokenizer.eos_token)

    input_ids = tokenized_meaning_representation + tokenized_human_reference + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))]

    labels = [-100 for _ in tokenized_meaning_representation] + tokenized_human_reference + [-100 for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))]

    tokenized_data.append({'input_ids': input_ids, 'labels': labels})
preprocessed_valid_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in test_dataset:
    tokenized_meaning_representation = tokenizer.encode(example['meaning_representation'] + tokenizer.bos_token)
    tokenized_human_reference = tokenizer.encode(' ' + example['human_reference'] + tokenizer.eos_token)

    input_ids = tokenized_meaning_representation + tokenized_human_reference + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))]

    labels = [-100 for _ in tokenized_meaning_representation] + tokenized_human_reference + [-100 for _ in range(max_seq_len - len(tokenized_meaning_representation) - len(tokenized_human_reference))]

    tokenized_data.append({'input_ids': input_ids, 'labels': labels})
preprocessed_test_dataset = Dataset.from_list(tokenized_data)

In [ ]:
# 원본 모델의 파라미터는 모두 고정하고, 추가할 어댑터 안에서만 학습가능한 파라미터를 두려고 함.
# 각 블록 안의 c_attn(W_Q, W_K, W_V를 합쳐둔 가중치), c_proj(W_O에 해당하는 가중치), c_fc(FFN 안의 W_fc1에 해당하는 가중치), c_proj(FFN 안의 W_fc2에 해당하는 가중치)는 뒤에서 EIDA.Linear_with_adapter 타입으로 교체할 때 requires_grad=False 설정이 이루어짐. 여기에 해당하지 않는 학습가능한 파라미터인 embedding과 layer normalization은 지금 고정함.
model.transformer.wte.requires_grad = False
model.transformer.wpe.requires_grad = False
for block in model.transformer.h: # 모델의 attention block 12개
    block.ln_1.requires_grad = False
    block.ln_2.requires_grad = False
model.ln_f.requires_grad = False


preprocessed_train_dataset = preprocessed_train_dataset.shuffle() # train set 순서 섞기
input_ids = torch.tensor(preprocessed_train_dataset['input_ids'])
labels = torch.tensor(preprocessed_train_dataset['labels'])

sample_inputs, sample_delta_outputs = EIDA.forward_gpt2(model, input_ids, labels, begin=0, end=256, batch_size=2, max_length=max_seq_len, p=0.01)


plane_inputs=[]
for i in range(4*12):
    plane_inputs.append(EIDA.PCA(sample_inputs[i], plane_dim=32))
del sample_inputs

plane_delta_outputs=[]
for i in range(6*12):
    plane_delta_outputs.append(EIDA.PCA(sample_delta_outputs[i], plane_dim=32))
del sample_delta_outputs